# Data Preprocessing Pipeline - Healthcare Stroke Dataset
Notebook ini digunakan untuk memuat, membersihkan, dan melakukan preprocessing dataset `healthcare-dataset-stroke-data.csv`[cite: 1].

In [7]:
import pandas as pd
import numpy as np

# Library untuk Preprocessing & Modeling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline



## 1. Load Data & Exploratory Data Analysis (EDA)
Memuat dataset dan mengecek gambaran umum serta missing values

In [8]:
# Memuat dataset
df = pd.read_csv('healthcare-dataset-stroke-data.csv')
# Display 5 baris pertama
print("=== 5 Baris Pertama Data ===")
display(df.head())

# Ringkasan info & missing values
print("\n=== Informasi Dataset ===")
df.info()

print("\n=== Jumlah Missing Values ===")
print(df.isnull().sum())

=== 5 Baris Pertama Data ===


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1



=== Informasi Dataset ===
<class 'pandas.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   str    
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   str    
 6   work_type          5110 non-null   str    
 7   Residence_type     5110 non-null   str    
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   str    
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), str(5)
memory usage: 634.7 KB

=== Jumlah Missing Values ===
id                     0
gender                 0
age                    0
hypertension           0
heart_disease 

## 2. Cleaning Awal & Pemisahan Fitur/Target
Menghapus kolom `id` dan outlier unik, lalu memisahkan data menjadi fitur ($X$) dan target ($y$)[cite: 1].

In [9]:
# Hapus kolom 'id' jika ada
if 'id' in df.columns:
    df = df.drop(columns=['id'])

# Hapus baris 'Other' pada gender (outlier sangat kecil)
df = df[df['gender'] != 'Other']

# Pemisahan Fitur (X) dan Target (y)
X = df.drop(columns=['stroke'])
y = df['stroke']

print(f"Distribusi Target (Stroke):\n{y.value_counts()}")

Distribusi Target (Stroke):
stroke
0    4860
1     249
Name: count, dtype: int64


## 3. Preprocessing Pipeline & Data Splitting
Membuat pipeline untuk otomatisasi penanganan *missing values* dan *scaling/encoding*, lalu membagi data menjadi *train* dan *test set*[cite: 1].

In [13]:
# Grouping kolom berdasarkan tipe data
num_cols = ['age', 'avg_glucose_level', 'bmi']
cat_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
binary_cols = ['hypertension', 'heart_disease']

# Pipeline Numerik (Imputasi Median + StandardScaler)
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline Kategorikal (Imputasi Modus + OneHotEncoder)
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine Pipeline menggunakan ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols),
    ('passthrough', 'passthrough', binary_cols)
])

# Train-Test Split (80:20) dengan Stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# Fit & Transform ke data
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

## 4. Format Hasil ke DataFrame
Mengubah kembali matriks NumPy hasil transformer ke bentuk `pandas.DataFrame` agar mudah di-inspect.

In [16]:
# 1. Ambil encoder dari dalam Pipeline 'cat'
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['encoder']

# 2. Ekstrak nama kolom baru hasil OneHotEncoder
encoded_cat_cols = list(cat_encoder.get_feature_names_out(cat_cols))

# 3. Gabungkan seluruh nama kolom (Numerik + One-Hot Categorical + Binary)
all_feature_names = num_cols + encoded_cat_cols + binary_cols

# 4. Buat DataFrame baru
X_train_df = pd.DataFrame(X_train_prepared, columns=all_feature_names)
X_test_df = pd.DataFrame(X_test_prepared, columns=all_feature_names)

print("=== Preprocessing Selesai ===")
print(f"Shape X_train olahan : {X_train_df.shape}")
display(X_train_df.head())

=== Preprocessing Selesai ===
Shape X_train olahan : (4087, 20)


,age,avg_glucose_level,bmi,gender_Female,gender_Male,ever_married_No,ever_married_Yes,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,hypertension,heart_disease
0,0.209397,-0.821221,0.549234,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1,-0.629845,-0.485884,-0.988900,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
2,-0.364822,0.302317,-0.769166,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,-0.232310,0.062342,0.497532,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,-1.292405,-0.527297,0.355351,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
